In [ ]:
import torch
import torch.nn as nn

# 1. LSTM Model Architecture
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        
        # LSTM layer: Input size 1, hidden layer size 32, number of layers 2
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        
        # Fully Connected output layer
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Initialize hidden and cell states with zeros
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        
        # LSTM forward pass
        out, (hn, cn) = self.lstm(x, (h0, c0))
        
        # We make a prediction by taking only the output of the last time step
        out = self.fc(out[:, -1, :])
        return out

# 2. GRU Model Architecture
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(GRUModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        
        # GRU layer
        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True)
        
        # Output layer
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # Initialize hidden state with zeros
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        
        # GRU forward pass
        out, hn = self.gru(x, h0)
        
        # Take the output of the last time step
        out = self.fc(out[:, -1, :])
        return out

# Define hyperparameters
input_dim = 1     # A single feature (Solar energy generation)
hidden_dim = 32   # Number of hidden layer units
layer_dim = 2     # Number of model layers
output_dim = 1    # The single value to predict (Next hour's generation)

# Set device (use GPU if available, e.g., RTX 3050)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Instantiate models and move them to the device
lstm_model = LSTMModel(input_dim, hidden_dim, layer_dim, output_dim).to(device)
gru_model = GRUModel(input_dim, hidden_dim, layer_dim, output_dim).to(device)

print("LSTM Model Architecture:\n", lstm_model)
print("\nGRU Model Architecture:\n", gru_model)